# Arabic Text Preprocessing for the AAFAQ Dataset

This notebook implements an Arabic text preprocessing pipeline for the AAFAQ dataset, which contains 5,009 Modern Standard Arabic question-answer pairs across 17 categories.

The preprocessing pipeline includes:
- Emoji removal
- Diacritics (Tashkeel) removal
- Tatweel removal
- Arabic letter normalization
- Punctuation, number, and special-character removal
- Extra-space removal
- Stopword removal
- Tokenization
- Stemming
- Lemmatization

Different tokenization, stemming, and lemmatization methods are compared to select the most suitable techniques for Arabic NLP.

## 1. Setup and Imports

In [ ]:
!pip install pyarabic emoji nltk scikit-learn pandas numpy qalsadi

In [2]:
import re
import emoji
import nltk
import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import qalsadi.lemmatizer as lm
from nltk.stem.isri import ISRIStemmer
from nltk.stem.snowball import SnowballStemmer
from nltk.stem import WordNetLemmatizer
import time

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

## 2. Dataset Loading and Overview

In [3]:
df = pd.read_csv('/content/AAFAQ_Dataset (1).csv')

In [4]:
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


In [5]:
print("data shape:")
df.shape

data shape:


(5009, 3)

In [6]:
print("Number of Categories:")
df['Category'].nunique()

Number of Categories:


17

In [7]:
print("Categories:")
df['Category'].unique()

Categories:


array(['التعليم', 'الاقتصاد والعمل', 'الصحة', 'التكنولوجيا',
       'البيئة والطاقة', 'الدين', 'الثقافة', 'الجغرافيا',
       'السياسة والقانون', 'التاريخ', 'العلوم', 'الترفيه', 'الرياضة',
       'السفر والسياحة', 'التطوع', 'علم الاجتماع', 'البيولوجيا'],
      dtype=object)

In [8]:
print("Null values:")
df.isnull().sum()

Null values:


,0
QuestionText,0
Category,0
Answer,0


## 3. Arabic Text Cleaning

In [ ]:
#Now we will create new columns so the original QuestionText and Answer stay unchanged
df['Clean_Question'] = df['QuestionText'].copy()
df['Clean_Answer'] = df['Answer'].copy()

df.head()

,QuestionText,Category,Answer,Clean_Question,Clean_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.


In [10]:
#We will start by removing the emojies from question and answer columns

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: emoji.replace_emoji(x, replace=''))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: emoji.replace_emoji(x, replace=''))

df.head()

,QuestionText,Category,Answer,Clean_Question,Clean_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.


In [11]:
#Now we will remove arabic diacritics (tashkeel)

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: araby.strip_tashkeel(x))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: araby.strip_tashkeel(x))

df.head()

,QuestionText,Category,Answer,Clean_Question,Clean_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,أيتكون الهواء أساسا من النيتروجين؟,الهواء يتكون أساسا من النيتروجين.


In [12]:
#Now we will remove any tatweel

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: araby.strip_tatweel(x))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: araby.strip_tatweel(x))

df.head()

,QuestionText,Category,Answer,Clean_Question,Clean_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,أيتكون الهواء أساسا من النيتروجين؟,الهواء يتكون أساسا من النيتروجين.


In [13]:
#Now we will normalize arabic letters

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: re.sub(r'[إأآا]','ا',str(x)))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: re.sub(r'[إأآا]','ا',str(x)))

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: re.sub(r'ؤ','و',str(x)))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: re.sub(r'ؤ','و',str(x)))

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: re.sub(r'ئ','ي',str(x)))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: re.sub(r'ئ','ي',str(x)))

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: re.sub(r'ى','ي',str(x)))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: re.sub(r'ى','ي',str(x)))

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: re.sub(r'ة','ه',str(x)))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: re.sub(r'ة','ه',str(x)))

df.head()

,QuestionText,Category,Answer,Clean_Question,Clean_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسه في السابق ام في الوقت الحالي؟,الدراسه في الوقت الحالي تعتبر افضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروه في مصر؟,القطن يعتبر من اهم المنتجات الزراعيه في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حيه دقيقه؟,البكتيريا تعرف بانها كاينات حيه دقيقه.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهواء اساسا من النيتروجين؟,الهواء يتكون اساسا من النيتروجين.


In [14]:
#Now we will remove any spcial charecters, numbers, or punctuation

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: re.sub(r'[^\u0600-\u06FF\s]', ' ', x))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: re.sub(r'[^\u0600-\u06FF\s]', ' ', x))

arabic_punctuations = r'[!@#$%^&*()_\-+=><؛×÷|\\:/،ـ؟.,{}\[\]~]'


df['Clean_Question'] = df['Clean_Question'].apply(lambda x: re.sub(arabic_punctuations, '', str(x)))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: re.sub(arabic_punctuations, '', str(x)))

df.head()

,QuestionText,Category,Answer,Clean_Question,Clean_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسه في السابق ام في الوقت الحالي,الدراسه في الوقت الحالي تعتبر افضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروه في مصر,القطن يعتبر من اهم المنتجات الزراعيه في مصر وي...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق,الشمس تصعد من الشرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حيه دقيقه,البكتيريا تعرف بانها كاينات حيه دقيقه
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهواء اساسا من النيتروجين,الهواء يتكون اساسا من النيتروجين


In [15]:
#Now we will remove any extra space

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()))

df.head()

,QuestionText,Category,Answer,Clean_Question,Clean_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسه في السابق ام في الوقت الحالي,الدراسه في الوقت الحالي تعتبر افضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروه في مصر,القطن يعتبر من اهم المنتجات الزراعيه في مصر وي...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق,الشمس تصعد من الشرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حيه دقيقه,البكتيريا تعرف بانها كاينات حيه دقيقه
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهواء اساسا من النيتروجين,الهواء يتكون اساسا من النيتروجين


In [16]:
#Now we will remove arabic stopwords

stop_words = set(stopwords.words('arabic'))

df['Clean_Question'] = df['Clean_Question'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))
df['Clean_Answer'] = df['Clean_Answer'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

df.head()


,QuestionText,Category,Answer,Clean_Question,Clean_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسه السابق ام الوقت الحالي,الدراسه الوقت الحالي تعتبر افضل بسبب توفر التك...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروه مصر,القطن يعتبر اهم المنتجات الزراعيه مصر ويعد الا...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس الشرق,الشمس تصعد الشرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حيه دقيقه,البكتيريا تعرف بانها كاينات حيه دقيقه
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهواء اساسا النيتروجين,الهواء يتكون اساسا النيتروجين


## 4. Tokenization

In [17]:
#Here we try tokenization using python split
start_time = time.time()

df['question_tokens_split'] = df['Clean_Question'].apply(lambda x: str(x).split())
df['answer_tokens_split'] = df['Clean_Answer'].apply(lambda x: str(x).split())

split_time = time.time() - start_time

print("Split runtime: ", split_time)
df[['Clean_Question', 'question_tokens_split']].head()

Split runtime:  0.12642693519592285


,Clean_Question,question_tokens_split
0,ايهما افضل الدراسه السابق ام الوقت الحالي,"[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]"
1,اليس القطن عماد الثروه مصر,"[اليس, القطن, عماد, الثروه, مصر]"
2,اتصعد الشمس الشرق,"[اتصعد, الشمس, الشرق]"
3,اتعرف البكتيريا بانها كاينات حيه دقيقه,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]"
4,ايتكون الهواء اساسا النيتروجين,"[ايتكون, الهواء, اساسا, النيتروجين]"


In [18]:
#Here we use tokenization using NLTK word_tokenize()
start_time = time.time()

df['question_tokens_nltk'] = df['Clean_Question'].apply(lambda x: word_tokenize(str(x)))
df['answer_tokens_nltk'] = df['Clean_Answer'].apply(lambda x: word_tokenize(str(x)))

word_tokenize_time = time.time() - start_time

print("NLTK runtime: ", word_tokenize_time)
df[['Clean_Question', 'question_tokens_nltk']].head()

NLTK runtime:  2.8420727252960205


,Clean_Question,question_tokens_nltk
0,ايهما افضل الدراسه السابق ام الوقت الحالي,"[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]"
1,اليس القطن عماد الثروه مصر,"[اليس, القطن, عماد, الثروه, مصر]"
2,اتصعد الشمس الشرق,"[اتصعد, الشمس, الشرق]"
3,اتعرف البكتيريا بانها كاينات حيه دقيقه,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]"
4,ايتكون الهواء اساسا النيتروجين,"[ايتكون, الهواء, اساسا, النيتروجين]"


As we can see the split is faster than the NLTK word_tokenize so we will select the split method for tokenization

## 5. Stemming

In [19]:
#Here we try stemming using ISRIStemmer
start_time = time.time()

isri = ISRIStemmer()

df['question_stemmed_isri'] = df['question_tokens_split'].apply(lambda x: [isri.stem(word) for word in x])
df['answer_stemmed_isri'] = df['answer_tokens_split'].apply(lambda x: [isri.stem(word) for word in x])

isri_time = time.time() - start_time

print("ISRI runtime: ", isri_time)
df[['question_tokens_split', 'question_stemmed_isri']].head(10)

ISRI runtime:  1.3820888996124268


,question_tokens_split,question_stemmed_isri
0,"[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]","[ايه, فضل, درس, سبق, ام, وقت, الحالي]"
1,"[اليس, القطن, عماد, الثروه, مصر]","[الس, قطن, عمد, ثره, مصر]"
2,"[اتصعد, الشمس, الشرق]","[صعد, شمس, شرق]"
3,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[عرف, كتر, بان, كين, حيه, دقق]"
4,"[ايتكون, الهواء, اساسا, النيتروجين]","[ايت, هوء, سسا, ترج]"
5,"[ايفرز, البنكرياس, الانسولين]","[فرز, كرياس, سول]"
6,"[ايتكون, الماء, الهيدروجين, والاوكسجين]","[ايت, ماء, هيدروج, كسج]"
7,"[ايوجد, فيتامين, سي, البرتقال]","[وجد, يتم, سي, رتقال]"
8,"[ايتنفس, السمك, باستخدام, الخياشيم]","[نفس, سمك, باستخدام, خياشيم]"
9,"[ايمكن, للنباتات, ان, تقوم, بعمليه, البناء, ال...","[ايم, نبت, ان, تقم, عمل, بنء, ضوي]"


In [20]:
#Here we try stemming using Snowball Stemmer

start_time = time.time()

snowball = SnowballStemmer('arabic')

df['question_stemmed_snowball'] = df['question_tokens_split'].apply(lambda x: [snowball.stem(word) for word in x])
df['answer_stemmed_snowball'] = df['answer_tokens_split'].apply(lambda x: [snowball.stem(word) for word in x])

snowball_time = time.time() - start_time

print("Snowball runtime: ", snowball_time)
df[['question_tokens_split', 'question_stemmed_snowball']].head(10)

Snowball runtime:  2.9952621459960938


,question_tokens_split,question_stemmed_snowball
0,"[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]","[ايهم, افضل, دراسه, سابق, ام, الو, حال]"
1,"[اليس, القطن, عماد, الثروه, مصر]","[اليس, قطن, عماد, ثروه, مصر]"
2,"[اتصعد, الشمس, الشرق]","[اتصعد, شمس, شرق]"
3,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[اتعرف, بكتير, بان, كاين, حيه, دقيق]"
4,"[ايتكون, الهواء, اساسا, النيتروجين]","[ايتك, هواء, اساس, نيتروج]"
5,"[ايفرز, البنكرياس, الانسولين]","[ايفرز, بنكرياس, انسول]"
6,"[ايتكون, الماء, الهيدروجين, والاوكسجين]","[ايتك, ماء, هيدروج, اوكسج]"
7,"[ايوجد, فيتامين, سي, البرتقال]","[ايوجد, يتام, سي, برتقال]"
8,"[ايتنفس, السمك, باستخدام, الخياشيم]","[ايتنفس, سمك, استخدام, خياشيم]"
9,"[ايمكن, للنباتات, ان, تقوم, بعمليه, البناء, ال...","[ايم, نبات, ان, تقوم, عمل, بناء, ضوي]"


As we can see snowball stemmer keeps more meaningful Arabic word forms than ISRIStemmer while ISRI produces moer aggressive root like stem. Tha's why we will use Snowball method for Stemming even though ISRI was faster.

## 6. Lemmatization

In [21]:
#Here we will uses WordNetLemmatizer

start_time = time.time()

wordnet_lemmatizer = WordNetLemmatizer()

df['question_lemma_wordnet'] = df['question_tokens_split'].apply(lambda x: [wordnet_lemmatizer.lemmatize(word) for word in x])
df['answer_lemma_wordnet'] = df['answer_tokens_split'].apply(lambda x: [wordnet_lemmatizer.lemmatize(word) for word in x])

wordnet_time = time.time() - start_time

print("WordNet runtime: ", wordnet_time)
df[['question_tokens_split', 'question_lemma_wordnet']].head()

WordNet runtime:  3.594682216644287


,question_tokens_split,question_lemma_wordnet
0,"[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]","[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]"
1,"[اليس, القطن, عماد, الثروه, مصر]","[اليس, القطن, عماد, الثروه, مصر]"
2,"[اتصعد, الشمس, الشرق]","[اتصعد, الشمس, الشرق]"
3,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]"
4,"[ايتكون, الهواء, اساسا, النيتروجين]","[ايتكون, الهواء, اساسا, النيتروجين]"


In [22]:
#Here we try lemmatization using Qalsadi

start_time = time.time()

lemmatizer = lm.Lemmatizer()

df['question_lemma_qalsadi'] = df['question_tokens_split'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
df['answer_lemma_qalsadi'] = df['answer_tokens_split'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

qalsadi_time = time.time() - start_time

print("Qalsadi runtime: ", qalsadi_time)
df[['question_tokens_split', 'question_lemma_qalsadi']].head()

Qalsadi runtime:  129.24723649024963


,question_tokens_split,question_lemma_qalsadi
0,"[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]","[وهم, فضل, الدراسه, سابق, ام, وقت, حال]"
1,"[اليس, القطن, عماد, الثروه, مصر]","[ليس, قطن, عماد, الثروه, مصر]"
2,"[اتصعد, الشمس, الشرق]","[اتصعد, شمس, شارق]"
3,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[اتعرف, بكتيريا, بان, كاينات, حي, دقيق]"
4,"[ايتكون, الهواء, اساسا, النيتروجين]","[ايتكون, هواء, اساسا, نيتروجين]"


As we can see, WordNet showed no effect on Arabic words, while Qalsadi provided meaningful Arabic lemmas. Therefor Qalsadi was chosen for lemmatization

In [23]:
df['Lemmatized_Question'] = df['question_lemma_qalsadi'].apply(lambda x: ' '.join(x))
df['Lemmatized_Answer'] = df['answer_lemma_qalsadi'].apply(lambda x: ' '.join(x))

df['Stemmed_Question'] = df['question_stemmed_snowball'].apply(lambda x: ' '.join(x))
df['Stemmed_Answer'] = df['answer_stemmed_snowball'].apply(lambda x: ' '.join(x))

df.head()

,QuestionText,Category,Answer,Clean_Question,Clean_Answer,question_tokens_split,answer_tokens_split,question_tokens_nltk,answer_tokens_nltk,question_stemmed_isri,...,question_stemmed_snowball,answer_stemmed_snowball,question_lemma_wordnet,answer_lemma_wordnet,question_lemma_qalsadi,answer_lemma_qalsadi,Lemmatized_Question,Lemmatized_Answer,Stemmed_Question,Stemmed_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسه السابق ام الوقت الحالي,الدراسه الوقت الحالي تعتبر افضل بسبب توفر التك...,"[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]","[الدراسه, الوقت, الحالي, تعتبر, افضل, بسبب, تو...","[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]","[الدراسه, الوقت, الحالي, تعتبر, افضل, بسبب, تو...","[ايه, فضل, درس, سبق, ام, وقت, الحالي]",...,"[ايهم, افضل, دراسه, سابق, ام, الو, حال]","[دراسه, الو, حال, تعتبر, افضل, سبب, توفر, تكنو...","[ايهما, افضل, الدراسه, السابق, ام, الوقت, الحالي]","[الدراسه, الوقت, الحالي, تعتبر, افضل, بسبب, تو...","[وهم, فضل, الدراسه, سابق, ام, وقت, حال]","[الدراسه, وقت, حال, اعتبر, فضل, سبب, توفر, تكن...",وهم فضل الدراسه سابق ام وقت حال,الدراسه وقت حال اعتبر فضل سبب توفر تكنولوجي ما...,ايهم افضل دراسه سابق ام الو حال,دراسه الو حال تعتبر افضل سبب توفر تكنولوج موار...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروه مصر,القطن يعتبر اهم المنتجات الزراعيه مصر ويعد الا...,"[اليس, القطن, عماد, الثروه, مصر]","[القطن, يعتبر, اهم, المنتجات, الزراعيه, مصر, و...","[اليس, القطن, عماد, الثروه, مصر]","[القطن, يعتبر, اهم, المنتجات, الزراعيه, مصر, و...","[الس, قطن, عمد, ثره, مصر]",...,"[اليس, قطن, عماد, ثروه, مصر]","[قطن, يعتبر, اهم, منتج, زراعيه, مصر, يعد, اعمد...","[اليس, القطن, عماد, الثروه, مصر]","[القطن, يعتبر, اهم, المنتجات, الزراعيه, مصر, و...","[ليس, قطن, عماد, الثروه, مصر]","[قطن, اعتبر, اهم, منتج, الزراعيه, مصر, أعاد, ا...",ليس قطن عماد الثروه مصر,قطن اعتبر اهم منتج الزراعيه مصر أعاد الاعمده ا...,اليس قطن عماد ثروه مصر,قطن يعتبر اهم منتج زراعيه مصر يعد اعمده رييسيه...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس الشرق,الشمس تصعد الشرق,"[اتصعد, الشمس, الشرق]","[الشمس, تصعد, الشرق]","[اتصعد, الشمس, الشرق]","[الشمس, تصعد, الشرق]","[صعد, شمس, شرق]",...,"[اتصعد, شمس, شرق]","[شمس, تصعد, شرق]","[اتصعد, الشمس, الشرق]","[الشمس, تصعد, الشرق]","[اتصعد, شمس, شارق]","[شمس, صعد, شارق]",اتصعد شمس شارق,شمس صعد شارق,اتصعد شمس شرق,شمس تصعد شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حيه دقيقه,البكتيريا تعرف بانها كاينات حيه دقيقه,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[البكتيريا, تعرف, بانها, كاينات, حيه, دقيقه]","[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[البكتيريا, تعرف, بانها, كاينات, حيه, دقيقه]","[عرف, كتر, بان, كين, حيه, دقق]",...,"[اتعرف, بكتير, بان, كاين, حيه, دقيق]","[بكتير, تعرف, بان, كاين, حيه, دقيق]","[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[البكتيريا, تعرف, بانها, كاينات, حيه, دقيقه]","[اتعرف, بكتيريا, بان, كاينات, حي, دقيق]","[بكتيريا, تعرف, بان, كاينات, حي, دقيق]",اتعرف بكتيريا بان كاينات حي دقيق,بكتيريا تعرف بان كاينات حي دقيق,اتعرف بكتير بان كاين حيه دقيق,بكتير تعرف بان كاين حيه دقيق
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهواء اساسا النيتروجين,الهواء يتكون اساسا النيتروجين,"[ايتكون, الهواء, اساسا, النيتروجين]","[الهواء, يتكون, اساسا, النيتروجين]","[ايتكون, الهواء, اساسا, النيتروجين]","[الهواء, يتكون, اساسا, النيتروجين]","[ايت, هوء, سسا, ترج]",...,"[ايتك, هواء, اساس, نيتروج]","[هواء, يتكو, اساس, نيتروج]","[ايتكون, الهواء, اساسا, النيتروجين]","[الهواء, يتكون, اساسا, النيتروجين]","[ايتكون, هواء, اساسا, نيتروجين]","[هواء, تك, اساسا, نيتروجين]",ايتكون هواء اساسا نيتروجين,هواء تك اساسا نيتروجين,ايتك هواء اساس نيتروج,هواء يتكو اساس نيتروج


## 7. Export Preprocessed Dataset

In [24]:
df[['QuestionText',
    'Clean_Question',
    'Category',
    'Answer',
    'Clean_Answer',
    'question_tokens_split',
    'answer_tokens_split',
    'question_stemmed_snowball',
    'answer_stemmed_snowball',
    'question_lemma_qalsadi',
    'answer_lemma_qalsadi',
    'Lemmatized_Question',
    'Lemmatized_Answer',
    'Stemmed_Question',
    'Stemmed_Answer']].to_csv('AAFAQ_preprocessed.csv', index=False, encoding='utf-8-sig')
